## 2.2 Limpeza de Dados

**Decisões e transformações aplicadas:**



- **Missing values:** Imputação com zero para variáveis numéricas e 'Desconhecido' para categóricas.

- **Outliers:** Detecção por IQR e substituição por NaN (poderá ser imputado ou removido conforme análise).

- **Inconsistências:** Padronização de siglas de UF, remoção de espaços e tratamento de duplicidades.

- **Padronização:** Uniformização de formatos de datas, textos e códigos (ex: datas para AAAA-MM-DD, textos para maiúsculas, remoção de acentos).


In [ ]:


from pathlib import Path

import pandas as pd

import numpy as np

import unicodedata



# Helper: remover acentos e normalizar texto para mapeamento de estados

def remove_accents(text):

    if not isinstance(text, str):

        return text

    return ''.join(c for c in unicodedata.normalize('NFD', text) if unicodedata.category(c) != 'Mn')



def normalize_text(text):

    if pd.isna(text):

        return ''

    return remove_accents(str(text)).strip().upper()



# Mapa de nomes de estados -> sigla (chaves normalizadas sem acentos)

estado_sigla_map = {

    'ACRE': 'AC','ALAGOAS': 'AL','AMAPA': 'AP','AMAZONAS': 'AM','BAHIA': 'BA',

    'CEARA': 'CE','DISTRITO FEDERAL': 'DF','ESPIRITO SANTO': 'ES','ESPÍRITO SANTO': 'ES',

    'GOIAS': 'GO','MARANHAO': 'MA','MATO GROSSO': 'MT','MATO GROSSO DO SUL': 'MS',

    'MINAS GERAIS': 'MG','PARA': 'PA','PARAIBA': 'PB','PARANA': 'PR','PERNAMBUCO': 'PE',

    'PIAUI': 'PI','RIO DE JANEIRO': 'RJ','RIO GRANDE DO NORTE': 'RN','RIO GRANDE DO SUL': 'RS',

    'RONDONIA': 'RO','RORAIMA': 'RR','SANTA CATARINA': 'SC','SAO PAULO': 'SP','SERGIPE': 'SE','TOCANTINS': 'TO'

}



def to_sigla(value):

    v = normalize_text(value)

    if not v:

        return 'ND'

    # já é sigla de 2 letras

    if len(v) == 2 and v.isalpha():

        return v

    # mapear por nome

    return estado_sigla_map.get(v, 'ND')



# 1) Limpeza Bolsa Atleta (Célula 1)

path_bolsa = Path('/home/lnk3x/Downloads/CieDados/FINAL/Bolsa Atleta Dezembro(Planilha1).csv')

df_atletas = pd.read_csv(path_bolsa, delimiter=';', encoding='utf-8-sig')

# Padronizar coluna UF e mapear nomes -> sigla

if 'UF' in df_atletas.columns:

    df_atletas['UF'] = df_atletas['UF'].astype(str).apply(lambda x: to_sigla(x))

else:

    df_atletas['UF'] = 'ND'



# Limpar valor pago

val_col = ' Valor Pago '

if val_col in df_atletas.columns:

    df_atletas['valor_pago'] = (

        df_atletas[val_col].astype(str)

        .str.replace('R$', '', regex=False)

        .str.replace('.', '', regex=False)

        .str.replace(',', '.', regex=False)

        .str.strip()

    )

    df_atletas['valor_pago'] = pd.to_numeric(df_atletas['valor_pago'], errors='coerce')

else:

    df_atletas['valor_pago'] = 0.0



# Imputação: valor pago faltante -> 0. UF faltante -> 'ND'

df_atletas['valor_pago'] = df_atletas['valor_pago'].fillna(0.0)



# Outliers: limitar ao percentil 1/99 para reduzir impacto extremo (registro)

lower_b = df_atletas['valor_pago'].quantile(0.01)

upper_b = df_atletas['valor_pago'].quantile(0.99)

outliers_b = ((df_atletas['valor_pago'] < lower_b) | (df_atletas['valor_pago'] > upper_b)).sum()

df_atletas['valor_pago'] = df_atletas['valor_pago'].clip(lower=lower_b, upper=upper_b)



# Agregados usados depois

uf_totals_bolsa = df_atletas.groupby('UF')['valor_pago'].sum().sort_index()

uf_counts = df_atletas.groupby('UF').size().sort_index()



# 2) Limpeza ATU (Célula 2) — Ensino Médio 3ª série

path_atu = Path('ATU_2025_BRASIL_REGIOES_UFS/ATU_BRASIL_REGIOES_UFS_2025.ods')

df_atu = pd.read_excel(path_atu, engine='odf', header=8)

if 'UNIDGEO' in df_atu.columns and 'MED_03_CAT_0' in df_atu.columns:

    df_atu['UNIDGEO'] = df_atu['UNIDGEO'].fillna('').astype(str).str.strip()

    df_atu['UF'] = df_atu['UNIDGEO'].apply(lambda x: to_sigla(x))

    df_atu['MED_03_CAT_0'] = pd.to_numeric(

        df_atu['MED_03_CAT_0'].astype(str).str.replace('.', '', regex=False).str.replace(',', '.', regex=False),

        errors='coerce'

    ).fillna(0)

else:

    df_atu = pd.DataFrame(columns=['UNIDGEO','MED_03_CAT_0','UF'])



# Outliers ATU: IQR capping

lower_a = df_atu['MED_03_CAT_0'].quantile(0.01) if not df_atu['MED_03_CAT_0'].empty else 0

upper_a = df_atu['MED_03_CAT_0'].quantile(0.99) if not df_atu['MED_03_CAT_0'].empty else 0

outliers_a = ((df_atu['MED_03_CAT_0'] < lower_a) | (df_atu['MED_03_CAT_0'] > upper_a)).sum() if 'MED_03_CAT_0' in df_atu.columns else 0

df_atu['MED_03_CAT_0'] = df_atu['MED_03_CAT_0'].clip(lower=lower_a, upper=upper_a)



# Agregado por UF (sigla)

uf_totals = df_atu.groupby('UF')['MED_03_CAT_0'].sum().sort_index()



# 3) Limpeza Tabela_Escola_2025 (Célula 3) — quadras

file_path = 'Tabela_Escola_2025.csv'

cols = ['SG_UF', 'IN_QUADRA_ESPORTES_COBERTA', 'IN_QUADRA_ESPORTES_DESCOBERTA']

chunksize = 100_000

frames = []

for chunk in pd.read_csv(file_path, sep=';', usecols=cols, chunksize=chunksize, encoding='latin1'):

    # padronizar SG_UF

    chunk['SG_UF'] = chunk['SG_UF'].astype(str).apply(lambda x: to_sigla(x))

    # garantir flags binárias (1 ou 0)

    for c in ['IN_QUADRA_ESPORTES_COBERTA', 'IN_QUADRA_ESPORTES_DESCOBERTA']:

        chunk[c] = pd.to_numeric(chunk[c], errors='coerce').fillna(0).astype(int)

        chunk[c] = chunk[c].apply(lambda v: 1 if v == 1 else 0)

    # filtrar apenas escolas com ao menos 1

    filtered = chunk[(chunk['IN_QUADRA_ESPORTES_COBERTA'] == 1) | (chunk['IN_QUADRA_ESPORTES_DESCOBERTA'] == 1)]

    frames.append(filtered)

if frames:

    result_clean = pd.concat(frames, ignore_index=True)

else:

    result_clean = pd.DataFrame(columns=cols)



soma_ee_por_uf = result_clean[result_clean['IN_QUADRA_ESPORTES_COBERTA'] == 1].groupby('SG_UF').size().sort_index()

soma_ef_por_uf = result_clean[result_clean['IN_QUADRA_ESPORTES_DESCOBERTA'] == 1].groupby('SG_UF').size().sort_index()



# 4) Resumo das transformações e registros

print('--- Resumo da Limpeza ---')

print(f"Bolsa Atleta: registros={len(df_atletas)}, UF ausentes={(df_atletas['UF']=='ND').sum()}, outliers_capados={outliers_b}")

print(f'ATU (Ensino Médio 3ª série): registros={len(df_atu)}, outliers_capados={outliers_a}')

print(f'Tabela Escola (quadras): registros após filtro={len(result_clean)}')



# Mostrar cabeçalhos limpos

print('\nAmostra df_atletas (limpo):')

print(df_atletas[['UF','valor_pago']].head())

print('\nAmostra df_atu (limpo):')

print(df_atu[['UF','MED_03_CAT_0']].head())

print('\nAmostra result_clean (quadras):')

print(result_clean.head())



# Garantir que as variáveis esperadas pelas próximas células existam

uf_totals_bolsa = uf_totals_bolsa.sort_index()

uf_counts = uf_counts.sort_index()

uf_totals = uf_totals.sort_index()

soma_ee_por_uf = soma_ee_por_uf.sort_index()

soma_ef_por_uf = soma_ef_por_uf.sort_index()

# --- Fim da célula de limpeza específica ---


--- Resumo da Limpeza ---
Bolsa Atleta: registros=9412, UF ausentes=0, outliers_capados=0
ATU (Ensino Médio 3ª série): registros=591, outliers_capados=5
Tabela Escola (quadras): registros após filtro=72800

Amostra df_atletas (limpo):
   UF  valor_pago
0  SP      2050.0
1  SP      2050.0
2  SP      2050.0
3  SP      2050.0
4  RJ      2050.0

Amostra df_atu (limpo):
   UF  MED_03_CAT_0
0  ND          27.0
1  ND         277.0
2  ND         189.0
3  ND         291.0
4  ND         292.0

Amostra result_clean (quadras):
  SG_UF  IN_QUADRA_ESPORTES_COBERTA  IN_QUADRA_ESPORTES_DESCOBERTA
0    RO                           1                              0
1    RO                           1                              0
2    RO                           1                              0
3    RO                           1                              0
4    RO                           1                              0


In [106]:
# As variáveis limpas já foram criadas na célula de limpeza.
# Basta usar as agregações limpas:
print('Total pago por UF:')
for uf, total in uf_totals_bolsa.items():
    print(f'{uf}: R$ {total:,.2f}')

print('\nNúmero de atletas por UF:')
for uf, count in uf_counts.items():
    print(f'{uf}: {count} atletas')

Total pago por UF:
AC: R$ 20,814.00
AL: R$ 99,956.00
AM: R$ 256,834.00
AP: R$ 170,812.00
BA: R$ 556,598.00
CE: R$ 383,808.00
DF: R$ 1,057,870.00
ES: R$ 554,504.00
GO: R$ 493,556.00
MA: R$ 260,236.00
MG: R$ 2,486,848.00
MS: R$ 375,212.00
MT: R$ 330,942.00
PA: R$ 267,854.00
PB: R$ 559,236.00
PE: R$ 871,130.00
PI: R$ 143,842.00
PR: R$ 2,325,784.00
RJ: R$ 3,835,162.00
RN: R$ 615,612.00
RO: R$ 32,802.00
RR: R$ 31,982.00
RS: R$ 1,677,024.00
SC: R$ 1,990,848.00
SE: R$ 178,690.00
SP: R$ 12,330,646.56
TO: R$ 62,730.00

Número de atletas por UF:
AC: 9 atletas
AL: 47 atletas
AM: 112 atletas
AP: 66 atletas
BA: 189 atletas
CE: 154 atletas
DF: 293 atletas
ES: 179 atletas
GO: 141 atletas
MA: 83 atletas
MG: 626 atletas
MS: 137 atletas
MT: 137 atletas
PA: 101 atletas
PB: 142 atletas
PE: 340 atletas
PI: 58 atletas
PR: 720 atletas
RJ: 1071 atletas
RN: 165 atletas
RO: 24 atletas
RR: 20 atletas
RS: 546 atletas
SC: 727 atletas
SE: 68 atletas
SP: 3218 atletas
TO: 39 atletas


In [107]:
# As variáveis limpas já foram criadas na célula de limpeza.
# Exibir por sigla de UF, não por nome extenso:
print('Total Ensino médio, 3° série (X) por UF (sigla):')
for uf, total in uf_totals.items():
    print(f'{uf}: {total}')

Total Ensino médio, 3° série (X) por UF (sigla):
AC: 3546.0
AL: 3749.0
AM: 3128.0
AP: 3300.0
BA: 4160.0
CE: 4364.0
DF: 3535.0
ES: 3855.0
GO: 3489.0
MA: 4320.0
MG: 3766.0
MS: 2461.0
MT: 2613.0
ND: 23163.0
PA: 3225.0
PB: 3061.0
PE: 3541.0
PI: 3544.0
PR: 3209.0
RJ: 2688.0
RN: 3873.0
RO: 3236.0
RR: 1853.0
RS: 2844.0
SC: 4111.0
SE: 3856.0
SP: 4010.0
TO: 3022.0


In [108]:
# As variáveis limpas já foram criadas na célula de limpeza.
# Basta usar as agregações limpas:
print('Soma de escolas com quadra coberta (EE) por UF:')
print(soma_ee_por_uf)

print('\nSoma de escolas com quadra descoberta (EF) por UF:')
print(soma_ef_por_uf)

Soma de escolas com quadra coberta (EE) por UF:
SG_UF
AC      165
AL      620
AM      818
AP      190
BA     2964
CE     2762
DF      643
ES     1278
GO     2026
MA      983
MG     6496
MS     1052
MT     1226
PA     1783
PB      912
PE     1942
PI      996
PR     4700
RJ     4785
RN      646
RO      564
RR      204
RS     2888
SC     2669
SE      473
SP    12242
TO      515
dtype: int64

Soma de escolas com quadra descoberta (EF) por UF:
SG_UF
AC      37
AL     253
AM     225
AP      26
BA    2446
CE     703
DF     435
ES     251
GO     806
MA     612
MG    2805
MS     307
MT     358
PA     838
PB     312
PE     716
PI     375
PR    1862
RJ    1846
RN     323
RO     117
RR      31
RS    2593
SC    1534
SE     167
SP    5735
TO     163
dtype: int64


In [109]:
# Comparação lado a lado das somas de quadras cobertas e descobertas por UF (dados limpos)
comparativo = pd.DataFrame({
    'Coberta (EE)': soma_ee_por_uf,
    'Descoberta (EF)': soma_ef_por_uf
}).fillna(0).astype(int)

print('Comparativo de escolas com quadra coberta e descoberta por UF:')
print(comparativo)

Comparativo de escolas com quadra coberta e descoberta por UF:
       Coberta (EE)  Descoberta (EF)
SG_UF                               
AC              165               37
AL              620              253
AM              818              225
AP              190               26
BA             2964             2446
CE             2762              703
DF              643              435
ES             1278              251
GO             2026              806
MA              983              612
MG             6496             2805
MS             1052              307
MT             1226              358
PA             1783              838
PB              912              312
PE             1942              716
PI              996              375
PR             4700             1862
RJ             4785             1846
RN              646              323
RO              564              117
RR              204               31
RS             2888             2593
SC          

In [110]:
# Comparação lado a lado das somas de quadras cobertas, descobertas, total de atletas, valor pago e ensino médio 3ª série por UF (dados limpos)
comparativo_full = pd.DataFrame({
    'Coberta (EE)': soma_ee_por_uf,
    'Descoberta (EF)': soma_ef_por_uf,
    'Atletas Bolsa': uf_counts,
    'Valor Pago Bolsa': uf_totals_bolsa,
    'Ensino Médio 3ª série': uf_totals
}).fillna(0)

# Ajustar tipos
comparativo_full['Coberta (EE)'] = comparativo_full['Coberta (EE)'].astype(int)
comparativo_full['Descoberta (EF)'] = comparativo_full['Descoberta (EF)'].astype(int)
comparativo_full['Atletas Bolsa'] = comparativo_full['Atletas Bolsa'].astype(int)
comparativo_full['Valor Pago Bolsa'] = comparativo_full['Valor Pago Bolsa'].astype(float)

print('Comparativo completo por UF:')
print(comparativo_full)

Comparativo completo por UF:
    Coberta (EE)  Descoberta (EF)  Atletas Bolsa  Valor Pago Bolsa  \
AC           165               37              9          20814.00   
AL           620              253             47          99956.00   
AM           818              225            112         256834.00   
AP           190               26             66         170812.00   
BA          2964             2446            189         556598.00   
CE          2762              703            154         383808.00   
DF           643              435            293        1057870.00   
ES          1278              251            179         554504.00   
GO          2026              806            141         493556.00   
MA           983              612             83         260236.00   
MG          6496             2805            626        2486848.00   
MS          1052              307            137         375212.00   
MT          1226              358            137         3309

In [111]:
# Exibir IDH_educação por UF
import pandas as pd

# Substitua pelo caminho correto do arquivo de IDH se necessário
# Exemplo: 'IDH.csv' deve conter colunas 'UF' e 'IDH_educacao'
df_idh = pd.read_csv('IDH.csv', encoding='utf-8')

# Padronizar UF para sigla
if 'UF' in df_idh.columns:
    df_idh['UF'] = df_idh['UF'].astype(str).str.strip().str.upper()
else:
    print('Coluna UF não encontrada na tabela de IDH.')

print('IDH Educação por UF:')
for uf, idh in df_idh.set_index('UF')['IDHMeducacao'].sort_index().items():
    print(f'{uf}: {idh}')

IDH Educação por UF:


KeyError: 'IDHMeducacao'